# 🚀 Huấn luyện Mô hình ABSA (Single NLI Model)

Notebook này huấn luyện MỘT mô hình duy nhất xử lý cả 4 khía cạnh bằng phương pháp NLI (Aspect-Based Sentiment Analysis).
Thay vì train 4 model rời rạc tốn RAM, mô hình sẽ học cách ghép Khía Cạnh và Bình Luận để dự đoán 1 nhãn duy nhất.

In [ ]:
!pip install -q transformers datasets scikit-learn accelerate
from google.colab import files
import pandas as pd
import torch
from sklearn.model_selection import train_test_split
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
from torch.utils.data import Dataset
import os

## 1. Tải Dữ liệu Lên

In [ ]:
print('Hãy tải lên 2 file: auto_labeled_cleaned_positive_reviews.csv và auto_labeled_cleaned_negative_reviews.csv')
uploaded = files.upload()

df_pos = pd.read_csv('auto_labeled_cleaned_positive_reviews.csv', encoding='utf-8-sig')
df_neg = pd.read_csv('auto_labeled_cleaned_negative_reviews.csv', encoding='utf-8-sig')
df = pd.concat([df_pos, df_neg], ignore_index=True)

# Chuyển đổi dữ liệu sang dạng NLI (Aspect - Comment)
aspect_mapping = {
    'Quality': 'chất lượng',
    'Price': 'giá cả',
    'Delivery': 'giao hàng',
    'Service': 'dịch vụ'
}

data = []
for idx, row in df.iterrows():
    for aspect, text_aspect in aspect_mapping.items():
        label = row[aspect]
        if label in [-1, 0, 1, 2]:
            data.append({'comment': str(row['cleaned_comment']), 'aspect_text': text_aspect, 'label': label})

df_nli = pd.DataFrame(data)
# Map nhãn cũ sang nhãn mới cho model: -1 -> 3, 0 -> 0, 1 -> 1, 2 -> 2
df_nli['label_mapped'] = df_nli['label'].map({-1: 3, 0: 0, 1: 1, 2: 2})
print(f'Tổng số mẫu huấn luyện (NLI): {len(df_nli)}')
print(df_nli['label_mapped'].value_counts())

## 2. Chuẩn bị Mô hình và Dataset

In [ ]:
MODEL_NAME = 'vinai/phobert-base-v2'
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

class NLIDataset(Dataset):
    def __init__(self, texts, aspects, labels, tokenizer, max_len=128):
        self.texts = texts
        self.aspects = aspects
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, item):
        encoding = self.tokenizer(
            str(self.aspects[item]),
            str(self.texts[item]),
            add_special_tokens=True,
            max_length=self.max_len,
            padding='max_length',
            truncation=True,
            return_attention_mask=True,
            return_tensors='pt'
        )
        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'labels': torch.tensor(self.labels[item], dtype=torch.long)
        }

X_train, X_val, asp_train, asp_val, y_train, y_val = train_test_split(
    df_nli['comment'].tolist(), df_nli['aspect_text'].tolist(), df_nli['label_mapped'].tolist(), test_size=0.1, random_state=42
)

train_dataset = NLIDataset(X_train, asp_train, y_train, tokenizer)
val_dataset = NLIDataset(X_val, asp_val, y_val, tokenizer)

## 3. Huấn luyện Mô hình

In [ ]:
from sklearn.metrics import accuracy_score, f1_score
def compute_metrics(pred):
    labels = pred.label_ids
    preds = pred.predictions.argmax(-1)
    return {'accuracy': accuracy_score(labels, preds), 'f1': f1_score(labels, preds, average='weighted')}

model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=4)

training_args = TrainingArguments(
    output_dir='./phobert-absa-final',
    num_train_epochs=3,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=64,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    fp16=True
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics
)

trainer.train()
trainer.save_model('./phobert-absa-final')
tokenizer.save_pretrained('./phobert-absa-final')
print("Huấn luyện thành công!")

## 4. Tải Mô hình về máy

In [ ]:
import shutil
shutil.make_archive('phobert-absa-final', 'zip', '.', 'phobert-absa-final')
files.download('phobert-absa-final.zip')